# Imports

In [1]:
import os
import scvelo as scv
import scanpy as sc
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import einops
import umap
import gseapy as gp
from gseapy import enrichr
import torch

from scbmlp.datasets import get_classification_datasets
from scbmlp.models import ScBMLPClassifier, Config

In [2]:
# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)  # For CUDA if available
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set scanpy settings for deterministic behavior
sc.settings.verbosity = 2  # Reduce verbosity
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [3]:
os.makedirs("figures/cell_type/", exist_ok=True)

# Set figure params for blog dimensions
fig_width = 800
title_fontsize = 18
legend_fontsize = 14

# Load data

In [4]:
class_key = "clusters"
n_genes = 10_000
layer = "spliced"  # same as adata.X
device = "cpu"
random_state = 42  # For reproducible results

adata = scv.datasets.pancreas()
adata = adata[:, ~adata.var_names.str.startswith(("Rpl", "Rps"))]  # remove ribosomal genes
sc.pp.normalize_total(adata, layer=layer)
sc.pp.log1p(adata, layer=layer)
sc.pp.highly_variable_genes(adata, subset=True, n_top_genes=n_genes, layer=layer)

train_dataset, val_dataset, _ = get_classification_datasets(
    adata,
    class_key=class_key,
    train_split=0.7,
    val_split=0.3,
    layer=layer,
    random_state=random_state,
    device=device,
)

normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:00)


## Visualize

In [5]:
# Make cell type colormap
unique_clusters = adata.obs["clusters"].unique()

# colors = px.colors.qualitative.Set1[:len(unique_clusters)]  # High contrast, scientific
# colors = px.colors.qualitative.Dark2[:len(unique_clusters)]  # Darker colors
# colors = px.colors.qualitative.T10[:len(unique_clusters)]    # Tableau colors
# colors = px.colors.qualitative.Pastel1[:len(unique_clusters)]  # Softer colors
colors = px.colors.qualitative.Plotly[:len(unique_clusters)]  # Original Plotly colors
# colors = sc.pl.palettes.default_28[:len(unique_clusters)]     # Scanpy default

# import plotly.colors as pc
# colors = pc.sample_colorscale(cmap, [i/(len(unique_clusters)-1) for i in range(len(unique_clusters))])

cluster_colors = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

In [6]:
fig = px.scatter(
    x=adata.obsm["X_umap"][:, 0],
    y=adata.obsm["X_umap"][:, 1],
    color=adata.obs[class_key],
    title="Cell Types in UMAP Space",
    labels={"x": "UMAP1", "y": "UMAP2"},
    hover_data=[adata.obs["clusters"]],
    width=fig_width,
    # Use the custom discrete colors defined above
    color_discrete_map=cluster_colors,
    category_orders={class_key: list(cluster_colors.keys())},
)
fig.update_traces(marker=dict(size=3))

# Apply custom font sizes from earlier variables
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=250, r=250, t=100, b=50),
    coloraxis_colorbar=dict(
        title="frequency", 
        title_font=dict(size=legend_fontsize),
        title_side="right"  # Position title on the right side of the colorbar
    )
)
# Axis title font tuning and remove ticks (UMAP coordinates are not meaningful)
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), showticklabels=False, ticks="")
fig.update_yaxes(title_font=dict(size=axis_title_size), showticklabels=False, ticks="")

fig.show()

# Save combined figure
# fig.write_html(f"figures/cell_type/cell_type_umap.html")
fig.write_image(f"figures/cell_type/cell_type_umap_v2.png", width=fig_width, height=400, scale=2)

# Train

In [7]:
n_classes = adata.obs[class_key].nunique()
d_hidden = 128
n_epochs = 100
lr = 1e-4
device = "cpu"
batch_size = 64  # no more (poor loss), no less (slower)
weight_decay = 0.35e-0

In [8]:
cfg = Config(
    d_input=n_genes,
    d_hidden=d_hidden,
    d_output=n_classes,
    n_epochs=n_epochs,
    lr=lr,
    device=device,
    batch_size=batch_size,
    weight_decay=weight_decay,
    bias=True,
    seed=random_state
)
model = ScBMLPClassifier(cfg)
train_losses, train_metrics, val_losses, val_metrics = model.fit(
    train_dataset, val_dataset,
)

Training for 100 epochs: 100%|██████████| 100/100 [01:10<00:00,  1.42it/s, train_acc=0.9922, train_loss=0.1345, val_acc=0.8929, val_loss=0.2946]


In [9]:
# Create combined plot with loss and accuracy subplots

# Prepare data
loss_df = pd.DataFrame({
    'Epoch': list(range(len(train_losses))) + list(range(len(val_losses))),
    'Loss': train_losses + val_losses,
    'Type': ['Train'] * len(train_losses) + ['Validation'] * len(val_losses)
})

metric_df = pd.DataFrame({
    'Epoch': list(range(len(train_metrics))) + list(range(len(val_metrics))),
    'Accuracy': train_metrics + val_metrics,
    'Type': ['Train'] * len(train_metrics) + ['Validation'] * len(val_metrics)
})

# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Training and Validation Loss', 'Training and Validation Accuracy'],
    vertical_spacing=0.15
)

# Colors shared across metrics
colors = {'Train': 'blue', 'Validation': 'red'}

# Add loss plot traces (these will own the legend entries)
for type_name in ['Train', 'Validation']:
    type_data = loss_df[loss_df['Type'] == type_name]
    fig.add_trace(
        go.Scatter(
            x=type_data['Epoch'],
            y=type_data['Loss'],
            mode='lines',
            name=type_name,               # Single legend entry per type
            legendgroup=type_name,
            showlegend=True,              # Only show legend for loss traces
            line=dict(color=colors[type_name], width=2),
            hovertemplate=f"Epoch %{{x}}<br>{type_name} Loss: %{{y:.4f}}<extra></extra>"
        ),
        row=1, col=1
    )

# Add accuracy plot traces (no new legend entries, grouped with loss) - now solid lines
for type_name in ['Train', 'Validation']:
    type_data = metric_df[metric_df['Type'] == type_name]
    fig.add_trace(
        go.Scatter(
            x=type_data['Epoch'],
            y=type_data['Accuracy'],
            mode='lines',
            name=type_name,               # Same name but suppressed in legend
            legendgroup=type_name,
            showlegend=False,
            line=dict(color=colors[type_name], width=2),
            hovertemplate=f"Epoch %{{x}}<br>{type_name} Accuracy: %{{y:.4f}}<extra></extra>"
        ),
        row=2, col=1
    )

# Update layout (dimensions & base styling)
fig.update_layout(
    height=500,
    width=fig_width,
    title_text="Training Progress: Loss and Accuracy",
    showlegend=True,
    legend=dict(title=None, orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=80, r=80, t=80, b=60)
)

# Axis labels
fig.update_xaxes(title_text="Epoch", row=2, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=2, col=1)

# Apply custom font sizes
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize))
)

# Axis title and tick font tuning
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot
fig.write_html("figures/cell_type/cell_type_training.html")
fig.write_image("figures/cell_type/cell_type_training.png", width=fig_width, height=500, scale=2)

In [10]:
# Get predictions on validation set to analyze error distribution
model.eval()
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = batch
        logits = model(inputs)
        predictions = torch.softmax(logits, dim=1)
        all_predictions.append(predictions.cpu())
        all_targets.append(targets.cpu())

# Concatenate all batches
predictions = torch.cat(all_predictions, dim=0)
targets = torch.cat(all_targets, dim=0)

# Get predicted classes
predicted_classes = torch.argmax(predictions, dim=1)
correct_predictions = (predicted_classes == targets).float()

# Calculate per-class accuracy
class_names = adata.obs[class_key].cat.categories
per_class_accuracy = []
per_class_counts = []

for class_idx in range(n_classes):
    class_mask = (targets == class_idx)
    if class_mask.sum() > 0:
        class_accuracy = correct_predictions[class_mask].mean().item()
        class_count = class_mask.sum().item()
    else:
        class_accuracy = 0.0
        class_count = 0
    per_class_accuracy.append(class_accuracy)
    per_class_counts.append(class_count)

In [11]:
# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Per-Class Accuracy',
        'Per-Class Sample Count', 
        'Prediction Confidence Distribution',
        'Confusion Matrix (Top Classes)'
    ],
    specs=[[{"type": "xy"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "xy"}]],
    horizontal_spacing=0.25,  # More horizontal spacing to prevent label collisions
    vertical_spacing=0.22     # More vertical spacing for better separation
)

# 1. Bar plot of per-class accuracy
fig.add_trace(
    go.Bar(x=class_names, y=per_class_accuracy, name="Per-Class Accuracy"),
    row=1, col=1
)

# 2. Bar plot of per-class sample count
fig.add_trace(
    go.Bar(x=class_names, y=per_class_counts, name="Sample Count"),
    row=1, col=2
)

# 3. Histogram of prediction confidence (max probability)
max_confidences = torch.max(predictions, dim=1)[0].numpy()
fig.add_trace(
    go.Histogram(x=max_confidences, nbinsx=50, name="Prediction Confidence"),
    row=2, col=1
)

# 4. Confusion matrix for most common classes (top 6)
from sklearn.metrics import confusion_matrix
most_common_classes = np.argsort(per_class_counts)[-6:]  # Top 6 most frequent classes
class_subset = np.isin(targets.numpy(), most_common_classes)
subset_targets = targets[class_subset].numpy()
subset_predictions = predicted_classes[class_subset].numpy()

if len(subset_targets) > 0:
    cm = confusion_matrix(subset_targets, subset_predictions, labels=most_common_classes)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig.add_trace(
        go.Heatmap(
            z=cm_normalized,
            x=[class_names[i] for i in most_common_classes],
            y=[class_names[i] for i in most_common_classes],
            colorscale='Blues',
            text=cm,
            texttemplate="%{text}",
            name="Confusion Matrix",
            showscale=True,  # Only this trace gets the colorbar
            colorbar=dict(  # Valid title specification
                title=dict(text="Norm. Freq.", side='right', font=dict(size=legend_fontsize))
            )
        ),
        row=2, col=2
    )

fig.update_layout(
    height=700,  # Increase height to accommodate better spacing
    width=fig_width,  # Increase width to accommodate better spacing
    title_text="Cell Type Classification Analysis",
    showlegend=False
)

# Update axis labels
# y-labels for left column
fig.update_yaxes(title_text="Accuracy", row=1, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_yaxes(title_text="True", row=2, col=2)

# x-labels for bottom row
fig.update_xaxes(title_text="Max Probability", row=2, col=1)
fig.update_xaxes(title_text="Predicted", row=2, col=2)

# Rotate x-axis labels for readability
fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=2)

# Apply custom font sizes
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize))
)

# Axis title and tick font tuning
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

# Restrict colorbar to vertical span of bottom-right subplot only
# (align length and vertical center to that subplot's y-domain)
try:
    ydom = fig.layout.yaxis4.domain  # bottom-right subplot y-domain
    xdom = fig.layout.xaxis4.domain  # bottom-right subplot x-domain
    for tr in fig.data:
        if isinstance(tr, go.Heatmap):
            tr.colorbar.len = ydom[1] - ydom[0]
            tr.colorbar.y = (ydom[0] + ydom[1]) / 2
            tr.colorbar.yanchor = 'middle'
            # Place colorbar just to the right of the subplot
            tr.colorbar.x = xdom[1] + 0.02
            tr.colorbar.xanchor = 'left'
            tr.colorbar.thickness = 14
except Exception as e:
    print(f"Colorbar adjustment warning: {e}")

fig.show()

# Save plot
fig.write_html("figures/cell_type/cell_type_model_analysis.html")
fig.write_image("figures/cell_type/cell_type_model_analysis.png", width=fig_width, height=700, scale=2)

In [ ]:
# Plot bias distributions to check if they are being used
px.histogram(model.left.bias).show()
px.histogram(model.right.bias).show()

# Weight interpretation

In [20]:
def get_marker_gene_lists(
    gene_names: np.ndarray,
    vecs: np.ndarray,
    n_modules: int = 1,
    n_top_genes: int = 50,
) -> np.ndarray:
    """Extract marker genes optimized for GO analysis."""
    gene_lists = []
    for i in range(n_modules):
        top_idxs = vecs[:,i].topk(n_top_genes).indices
        top_genes = gene_names[top_idxs].tolist()
        bottom_idxs = (-vecs[:,i]).topk(n_top_genes).indices
        bottom_genes = gene_names[bottom_idxs].tolist()
        gene_lists.append([top_genes, bottom_genes])
    return np.array(gene_lists)

def decompose_gene_weights(model, out_idx):
    """Perform eigendecomposition of bilinear weights for a specific output."""
    q = einops.einsum(model.w_p[out_idx], model.w_l, model.w_r, "hid, hid in1, hid in2 -> in1 in2")
    q = 0.5 * (q + q.T)  # symmetrize
    vals, vecs = torch.linalg.eigh(q)
    vals = vals.flip([0])
    vecs = vecs.flip([1])
    return vals, vecs


def print_gene_modules(gene_name, gene_lists, n_modules=3):
    """Print gene modules for a specific gene."""
    print("="*20, gene_name, "="*20)
    for module_idx in range(n_modules):
        print("="*20, "Module", module_idx, "="*20)
        print(f"Positive genes: {gene_lists[module_idx,0,:10]}...")
        print(f"Negative genes: {gene_lists[module_idx,1,:10]}...")


def analyze_go_terms(gene_lists, n_modules=3, n_results=5):
    """Perform GO term analysis on gene modules."""
    print("\n" + "="*50)
    print("GO ANALYSIS")
    print("="*50)

    results_cols = ["Term", "Genes", "Gene_set", "Adjusted P-value"]

    for module_idx in range(n_modules):
        print("="*10, "Module", module_idx, "="*10)

        # Use Enrichr with optimized gene sets for pancreatic development
        for i in range(2):
            side = "Positive" if i == 0 else "Negative"
            print(f"\n--- {side} genes ---")
            enr = gp.enrichr(
                gene_list=gene_lists[module_idx, i].tolist(),
                gene_sets=[
                    "GO_Biological_Process_2023",
                    "GO_Molecular_Function_2023",
                    "GO_Cellular_Component_2023", 
                    "KEGG_2019_Mouse",
                    "Reactome_2022",
                    "MSigDB_Hallmark_2020",
                    "WikiPathways_2019_Mouse"
                ],
                organism="mouse",
            )
            # Filter and combine results from all gene sets
            all_results = enr.results[enr.results['Adjusted P-value'] <= 0.05].copy()
            if len(all_results) > 0:
                # Sort by p-value and show top results
                all_results = all_results.sort_values('Adjusted P-value')
                display(all_results.head(n_results)[results_cols])
            else:
                print(f"No significant results found (p < 0.05)")

In [24]:
# Shared parameters for class module analysis
n_modules = 3
n_top_genes = int(0.01*n_genes)  # top 1% of genes
gene_names = adata.var_names.values
n_results = 15

In [30]:
target_types = ["Ductal", "Ngn3 high EP", "Alpha", "Beta"]
type2idx = {t: int(np.where(class_names == t)[0][0]) for t in target_types}

## Ductal

In [31]:
cell_type = "Ductal"

In [33]:
# # Decompose weights and extract gene modules
# class_idx = type2idx[cell_type]
# vals, vecs = decompose_class_weights(model, class_idx)
# gene_lists = get_marker_gene_lists(
#     gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
# )

# Print gene modules
print_gene_modules(f"{cell_type}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Ductal ====================
==================== Module 0 ====================
Positive genes: ['Gcg' 'Ttr' 'Malat1' 'Gnas' 'Pyy' 'Tmem27' 'Eef1a1' 'Slc25a5' 'Cpe'
 'Chgb']...
Negative genes: ['Ghrl' 'Sst' 'Arg1' 'Pdx1' 'Npepl1' 'Enho' 'Gng12' 'Gm27033' 'Ins2'
 'Mfap4']...
==================== Module 1 ====================
Positive genes: ['Pyy' 'Iapp' 'Ghrl' 'Rbp4' 'Scgn' 'Pcsk2' 'Fam183b' 'Nnat' 'Tmem27'
 'Isl1']...
Negative genes: ['Spp1' 'Neurog3' 'Sox4' 'Mdk' 'Clu' 'Cd24a' 'Zfos1' 'Sparc' 'Vim' 'Btg2']...
==================== Module 2 ====================
Positive genes: ['Ghrl' 'Ttr' 'Spp1' 'Clu' 'Maged2' 'Gcg' 'Serpina1c' 'Pyy' 'Id1' 'Rbp4']...
Negative genes: ['Chgb' 'Chga' 'Cck' 'Fev' 'Map1b' 'Hmgn3' 'Krt7' 'Jun' 'Hhex' 'Tubb3']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1662,Pancreas Beta Cells,CHGA;PAX6;GCG;IAPP;NEUROG3,MSigDB_Hallmark_2020,0.000066
1663,KRAS Signaling Up,PCSK1N;PEG3;ID2;TSPAN7;SPP1;CPE;SCG3;ETV1,MSigDB_Hallmark_2020,0.000151
1247,Platelet Degranulation R-HSA-114608,TTR;TMSB4X;CD9;SCG3;PFN1;CALM1;CLU,Reactome_2022,0.000838
1248,Response To Elevated Platelet Cytosolic Ca2+ R...,TTR;TMSB4X;CD9;SCG3;PFN1;CALM1;CLU,Reactome_2022,0.000838
1249,Hemostasis R-HSA-109582,DOCK11;TTR;TMSB4X;TSPAN7;GNAS;CD9;SCG3;ATP2B1;...,Reactome_2022,0.004202
1250,"Platelet Activation, Signaling And Aggregation...",TTR;TMSB4X;CD9;SCG3;PFN1;CALM1;CLU;GNAI2,Reactome_2022,0.004276
1110,Estrogen signaling pathway,HSPA8;HSP90AA1;KRT18;GNAS;CALM1;GNAI2,KEGG_2019_Mouse,0.004891
1111,Gap junction,TUBA1B;TUBB5;TUBA1A;GNAS;GNAI2,KEGG_2019_Mouse,0.004891
1251,Protein Methylation R-HSA-8876725,EEF1A1;HSPA8;CALM1,Reactome_2022,0.006505
1112,Protein processing in endoplasmic reticulum,HSPA8;HSP90AA1;SSR4;SSR2;SEC61B;UBQLN2,KEGG_2019_Mouse,0.007733



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
900,Maturity onset diabetes of the young,INS1;HHEX;INS2;PDX1;PAX4;MNX1,KEGG_2019_Mouse,5.473700e-07
0,Positive Regulation Of Insulin Secretion (GO:0...,NNAT;PDX1;GHRL;SOX4,GO_Biological_Process_2023,1.909343e-02
1,Regulation Of Insulin Secretion (GO:0050796),NNAT;PDX1;GHRL;SOX4;ADRA2A,GO_Biological_Process_2023,1.909343e-02
2,Positive Regulation Of Peptide Hormone Secreti...,NNAT;PDX1;GHRL;SOX4,GO_Biological_Process_2023,1.909343e-02
1372,Pancreas Beta Cells,SST;PDX1;PAX4,MSigDB_Hallmark_2020,3.771702e-02


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
1177,Maturity onset diabetes of the young,PAX4;NKX6-1;FOXA3;NEUROG3;FOXA2,KEGG_2019_Mouse,0.000028
0,Regulation Of Transcription By RNA Polymerase ...,APP;SMARCD2;HMGB2;ENO1;LITAF;DNAJB1;EPCAM;SOX9...,GO_Biological_Process_2023,0.000108
1311,Regulation Of Beta-Cell Development R-HSA-186712,PAX4;NKX6-1;FOXA3;FOXA2;NEUROG3,Reactome_2022,0.000538
1604,Apoptosis,APP;JUN;BTG2;KRT18;HMGB2;TXNIP;CLU,MSigDB_Hallmark_2020,0.000739
1605,Pancreas Beta Cells,PAX4;NKX6-1;FOXA2;NEUROG3,MSigDB_Hallmark_2020,0.001054
1178,Tight junction,CLDN6;JUN;CDK4;ACTN1;AMOTL2;F11R;MYL12A,KEGG_2019_Mouse,0.001395
1606,Androgen Response,KRT19;ACTN1;KRT8;DBI;MYL12A,MSigDB_Hallmark_2020,0.002191
1607,TNF-alpha Signaling via NF-kB,JUN;BTG2;PLK2;FJX1;LITAF;IER2,MSigDB_Hallmark_2020,0.005704
1649,Spinal Cord Injury WP2432,BTG2;CDK4;TRP53;SOX9;VIM,WikiPathways_2019_Mouse,0.009195
1650,Novel Jun-Dmp1 Pathway WP3654,JUN;CDK4;TRP53,WikiPathways_2019_Mouse,0.009572


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
937,Endoplasmic Reticulum Lumen (GO:0005788),PDIA3;LRPAP1;HSPA5;CTSZ;GCG;PDIA6;HSP90B1;SPP1...,GO_Cellular_Component_2023,1.590530e-08
1046,Protein processing in endoplasmic reticulum,PDIA3;LMAN1;HSPA5;SSR4;RPN2;SSR1;CALR;PDIA6;HS...,KEGG_2019_Mouse,7.620512e-08
938,Intracellular Organelle Lumen (GO:0070013),PDIA3;SPARC;HSPA5;CTSZ;GCG;PDIA6;HSP90B1;GCSH;...,GO_Cellular_Component_2023,2.166579e-06
939,Secretory Granule Lumen (GO:0034774),GRN;SPARC;TTR;ARG1;CTSZ;GCG;MAGED2;QSOX1;GHRL;...,GO_Cellular_Component_2023,2.166579e-06
1181,Response To Elevated Platelet Cytosolic Ca2+ R...,CHID1;SPARC;TTR;HSPA5;ANXA5;TAGLN2;MAGED2;QSOX...,Reactome_2022,2.596031e-06
1180,Platelet Degranulation R-HSA-114608,CHID1;SPARC;TTR;HSPA5;ANXA5;TAGLN2;MAGED2;QSOX...,Reactome_2022,2.596031e-06
1467,IL-2/STAT5 Signaling,CDKN1C;MUC1;CCND3;CCND2;S100A1;CD81;CTSZ;SPP1;...,MSigDB_Hallmark_2020,2.918379e-05
940,Endocytic Vesicle Lumen (GO:0071682),SPARC;CTSL;CALR;HSP90B1,GO_Cellular_Component_2023,8.987870e-05
1182,Aberrant Regulation Of Mitotic G1/S Transition...,CDKN1C;CDKN1A;CCND3;CCND2,Reactome_2022,1.274184e-04
1468,mTORC1 Signaling,FKBP2;CDKN1A;SDF2L1;HSPA5;SERPINH1;SSR1;CALR;H...,MSigDB_Hallmark_2020,1.546863e-04



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1608,Pancreas Beta Cells,CHGA;SST;PDX1;PAX4;PAX6;NKX6-1,MSigDB_Hallmark_2020,0.000001
1049,Maturity onset diabetes of the young,HHEX;PDX1;PAX4;PAX6;NKX6-1,KEGG_2019_Mouse,0.000028
1609,TNF-alpha Signaling via NF-kB,EGR1;MARCKS;JUN;BTG2;SQSTM1;PNRC1,MSigDB_Hallmark_2020,0.005239
1610,Hypoxia,JUN;CITED2;SIAH2;PDK3;GLRX;PNRC1,MSigDB_Hallmark_2020,0.005239
1611,UV Response Up,DNAJA1;BTG2;IGFBP2;CCK;SQSTM1,MSigDB_Hallmark_2020,0.009216
803,Histone Acetyltransferase Binding (GO:0035035),EGR1;CITED2;PAX6,GO_Molecular_Function_2023,0.012149
1612,mTORC1 Signaling,GSK3B;BTG2;PSMC2;GLRX;SQSTM1,MSigDB_Hallmark_2020,0.020608
804,Ubiquitin Protein Ligase Binding (GO:0031625),DNAJA1;HSPA8;GSK3B;JUN;MAP1LC3A;SLC25A5;SQSTM1,GO_Molecular_Function_2023,0.032181
805,Ubiquitin-Like Protein Ligase Binding (GO:0044...,DNAJA1;HSPA8;GSK3B;JUN;MAP1LC3A;SLC25A5;SQSTM1,GO_Molecular_Function_2023,0.032181
806,Sequence-Specific DNA Binding (GO:0043565),EGR1;KDM5B;JUN;HHEX;JUND;FEV;PAX4;PDX1;PAX6;ME...,GO_Molecular_Function_2023,0.035244


In [47]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Ensure we have a target cell type and its eigen decomposition
try:
    cell_type  # from earlier cell (e.g., "Ductal")
except NameError:
    cell_type = "Ductal"

# Mapping from type name to class index (ensure it exists)
if 'type2idx' not in globals():
    # Rebuild from adata if missing
    class_names = adata.obs[class_key].cat.categories
    type2idx = {t: int(np.where(class_names == t)[0][0]) for t in class_names}

class_idx = type2idx[cell_type]

# Compute eigendecomposition for this target class if not already present
if 'vals' not in globals() or 'vecs' not in globals() or vecs.shape[1] < 3:
    vals, vecs = decompose_gene_weights(model, class_idx)

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
cell_type_labels = [class_names[i] for i in labels_np]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], cell_type_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Cell Type': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_cell_types = sorted(plot_df['Cell Type'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_cell_types):
        ctype_scores = module_df[module_df['Cell Type'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Cell Type: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Cell Type Distributions Along Top 3 Modules for {cell_type} Cells"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & cell types</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=80, r=40, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
# fig.write_html(f"figures/cell_type/{cell_type}_modules.html")
fig.write_image(f"figures/cell_type/{cell_type}_modules.png", width=fig_width, height=800, scale=2)

In [54]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/cell_type/cell_type_{freq_idx}_eigvals.html")
fig.write_image(f"figures/cell_type/{cell_type}_eigvals.png", width=fig_width, height=300, scale=2)

## Ngn3 high EP

In [57]:
cell_type = "Ngn3 high EP"

In [59]:
# Decompose weights and extract gene modules
class_idx = type2idx[cell_type]
vals, vecs = decompose_class_weights(model, class_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{cell_type}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Ngn3 high EP ====================
==================== Module 0 ====================
Positive genes: ['Neurog3' 'Gcg' '8430408G22Rik' 'Amotl2' 'Gast' 'Arx' 'Btbd17' 'Tmem171'
 'Gadd45a' 'Rasd1']...
Negative genes: ['Sst' 'Rbp4' 'Eef1a1' 'Hhex' 'Dlk1' 'Malat1' 'Cd24a' 'Isl1' 'Ssr2'
 'Atp1b1']...
==================== Module 1 ====================
Positive genes: ['Spp1' 'Clu' 'Sparc' 'Mt1' 'Mgst1' 'Dbi' 'H19' 'Cldn3' '1700011H14Rik'
 'Mt2']...
Negative genes: ['Isl1' 'Cck' 'Cdkn1a' 'Aplp1' 'Rbp4' 'Bex2' 'Chga' 'Ghrl' 'Hmgn3' 'Emb']...
==================== Module 2 ====================
Positive genes: ['Nkx6-1' 'Pdx1' 'Fev' 'Chgb' 'Ube2e3' 'Krt7' 'Npepl1' 'Tubb3' 'Pax4'
 'Btg2']...
Negative genes: ['Ghrl' 'Pyy' 'Gcg' 'Rbp4' 'Iapp' 'Tmem27' 'Ttr' 'Maged2' 'Gpx3' 'Hspa5']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1108,Regulation Of Gene Expression In Endocrine-Com...,PAX4;INSM1;NEUROG3,Reactome_2022,0.000336
1387,Pancreas Beta Cells,PAX4;GCG;INSM1;NEUROG3,MSigDB_Hallmark_2020,0.001639
914,Bicellular Tight Junction (GO:0005923),MARVELD2;CLDN9;WNK3;TBCD;AMOTL2,GO_Cellular_Component_2023,0.004768
915,Tight Junction (GO:0070160),MARVELD2;CLDN9;WNK3;TBCD;AMOTL2,GO_Cellular_Component_2023,0.004768
916,Apical Junction Complex (GO:0043296),MARVELD2;CLDN9;WNK3;TBCD;AMOTL2,GO_Cellular_Component_2023,0.005889
1109,Regulation Of Beta-Cell Development R-HSA-186712,PAX4;INSM1;FOXA3;NEUROG3,Reactome_2022,0.007211
771,E-box Binding (GO:0070888),NEUROD2;ATOH8;ASCL1;NEUROG3,GO_Molecular_Function_2023,0.017555
1110,VLDL Clearance R-HSA-8964046,APOC1;APOBR,Reactome_2022,0.022793
0,Regulation Of p38MAPK Cascade (GO:1900744),DUSP10;GADD45A;DAB2IP;GADD45G,GO_Biological_Process_2023,0.026392
1006,Maturity onset diabetes of the young,PAX4;FOXA3;NEUROG3,KEGG_2019_Mouse,0.033163



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1478,Protein processing in endoplasmic reticulum,PDIA3;HSPA8;HSPA5;SSR4;SSR2;CALR;SEC61B;PDIA6;...,KEGG_2019_Mouse,0.000024
0,Regulation Of Cell Population Proliferation (G...,JUN;JUND;TSC22D1;CD81;DYNLL1;CLU;MEIS2;HHEX;SS...,GO_Biological_Process_2023,0.000049
1479,cAMP signaling pathway,RAP1B;JUN;SST;GNAS;GHRL;SOX9;FOS;ATP1B1;CALM1,KEGG_2019_Mouse,0.000106
1,Negative Regulation Of Protein-Containing Comp...,HSPA8;HSPA5;IAPP;HMGB1;CLU;ISL1,GO_Biological_Process_2023,0.000286
1173,Cadherin Binding (GO:0045296),HSPA8;RANBP1;LDHA;KRT18;HSPA5;EPCAM;TAGLN2;PFN...,GO_Molecular_Function_2023,0.000392
1172,Cell-Cell Adhesion Mediator Activity (GO:0098632),KRT18;EPCAM;EMB;CD47;S100A11,GO_Molecular_Function_2023,0.000392
2134,Androgen Response,CCND1;TSC22D1;ADAMTS1;KRT8;DBI;MYL12A,MSigDB_Hallmark_2020,0.000482
1346,Secretory Granule Lumen (GO:0034774),EEF1A1;HSPA8;TTR;NPC2;ARG1;GHRL;HMGB1;ALDOA;CL...,GO_Cellular_Component_2023,0.000547
2178,Spinal Cord Injury WP2432,EGR1;CCND1;ARG1;SOX9;FOS;CD47,WikiPathways_2019_Mouse,0.000662
2137,Hypoxia,JUN;LDHA;HSPA5;MIF;FOS;ALDOA;GAPDH,MSigDB_Hallmark_2020,0.000723


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
1501,Pancreas Beta Cells,CHGA;ABCC8;SST;PAX6;ISL1;FOXA2;NEUROG3,MSigDB_Hallmark_2020,3.700936e-08
1170,Peptide Hormone Metabolism R-HSA-2980736,CHGA;FFAR4;CPE;PAX6;GHRL;MBOAT4;ISL1,Reactome_2022,1.030159e-04
797,Hormone Activity (GO:0005179),PYY;SST;PPY;CCK;GHRL;CHGB,GO_Molecular_Function_2023,2.626835e-04
798,Adrenergic Receptor Binding (GO:0031690),UCHL1;APLP1;GNAS,GO_Molecular_Function_2023,3.671463e-03
0,Axon Development (GO:0061564),APP;MAP1B;APLP1;CCK;ISL1;NEUROG3,GO_Biological_Process_2023,8.238737e-03
799,Neuropeptide Activity (GO:0160041),PYY;PPY;CCK,GO_Molecular_Function_2023,1.007922e-02
800,Neuropeptide Hormone Activity (GO:0005184),PYY;PPY;CCK,GO_Molecular_Function_2023,1.007922e-02
3,Positive Regulation Of Peptide Hormone Secreti...,GLUD1;FFAR4;GHRL;ISL1,GO_Biological_Process_2023,1.475992e-02
2,Regulation Of Insulin Secretion (GO:0050796),CHGA;GLUD1;GHRL;ISL1;FOXA2,GO_Biological_Process_2023,1.475992e-02
1,Cell Morphogenesis Involved In Neuron Differen...,APP;MAP1B;APLP1;CCK;ISL1,GO_Biological_Process_2023,1.475992e-02


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1019,Maturity onset diabetes of the young,HHEX;PDX1;PAX4;NKX6-1;MNX1;NEUROG3,KEGG_2019_Mouse,5.364226e-07
1761,Pancreas Beta Cells,CHGA;MAFB;PDX1;PAX4;NKX6-1;NEUROG3,MSigDB_Hallmark_2020,1.616519e-06
1166,Developmental Biology R-HSA-1266738,VASP;HSPA8;JUN;SIAH2;KRT8;PAX4;KRT7;GSPT1;KRT1...,Reactome_2022,3.282891e-03
937,Nucleus (GO:0005634),CASZ1;BTG2;SMARCD2;CITED2;GTF2B;CELF3;HMGB2;NE...,GO_Cellular_Component_2023,4.791859e-03
1763,Hypoxia,JUN;CITED2;SIAH2;PDK3;GLRX;PNRC1,MSigDB_Hallmark_2020,6.084358e-03
1762,TNF-alpha Signaling via NF-kB,EGR1;JUN;BTG2;GADD45A;PNRC1;IER2,MSigDB_Hallmark_2020,6.084358e-03
1797,Spinal Cord Injury WP2432,EGR1;PPP3CA;BTG2;GADD45A;CDK4,WikiPathways_2019_Mouse,9.055640e-03
1798,Novel Jun-Dmp1 Pathway WP3654,JUN;CDK4;MDM2,WikiPathways_2019_Mouse,9.426605e-03
0,Cellular Response To Glucose Stimulus (GO:0071...,PDX1;PDK3;NKX6-1;SOX4,GO_Biological_Process_2023,1.930562e-02
1800,Ptf1a related regulatory pathway WP201,PDX1;NKX6-1,WikiPathways_2019_Mouse,2.147990e-02



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
959,Protein processing in endoplasmic reticulum,PDIA3;HSPA5;SSR4;RPN2;TUSC3;SSR2;RPN1;PDIA6;HS...,KEGG_2019_Mouse,3.906034e-13
1365,Pancreas Beta Cells,PCSK2;SCGN;ABCC8;GCG;IAPP;ISL1,MSigDB_Hallmark_2020,1.930842e-06
873,Endoplasmic Reticulum Lumen (GO:0005788),PDIA3;LRPAP1;HSPA5;CTSZ;CANX;GCG;GHRL;CALR;PDI...,GO_Cellular_Component_2023,1.551933e-05
1366,mTORC1 Signaling,FKBP2;CDKN1A;SDF2L1;HSPA5;RPN1;CANX;SSR1;CALR;...,MSigDB_Hallmark_2020,1.557918e-05
1075,Peptide Hormone Metabolism R-HSA-2980736,PCSK2;ANPEP;CTSZ;GCG;GHRL;MBOAT4;ISL1,Reactome_2022,9.025562e-05
960,Thyroid hormone synthesis,TTR;HSPA5;GPX3;CANX;PDIA4;HSP90B1,KEGG_2019_Mouse,1.011280e-04
874,Endoplasmic reticulum-Golgi Intermediate Compa...,LMAN1;TMED10;CTSZ;TMED3;CALR,GO_Cellular_Component_2023,2.620393e-04
1079,Metabolism Of Proteins R-HSA-392499,PDIA3;PCSK2;SSR4;RPN2;TUSC3;SSR2;RPN1;CTSZ;GCG...,Reactome_2022,8.598334e-04
1078,Asparagine N-linked Glycosylation R-HSA-446203,PDIA3;LMAN1;RPN2;TUSC3;RPN1;CTSZ;CANX;TMED3;CALR,Reactome_2022,8.598334e-04
1077,"Antigen Presentation: Folding, Assembly, Pepti...",PDIA3;HSPA5;CANX;CALR,Reactome_2022,8.598334e-04


In [60]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

class_idx = type2idx[cell_type]

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
cell_type_labels = [class_names[i] for i in labels_np]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], cell_type_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Cell Type': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_cell_types = sorted(plot_df['Cell Type'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_cell_types):
        ctype_scores = module_df[module_df['Cell Type'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Cell Type: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Cell Type Distributions Along Top 3 Modules for {cell_type} Cells"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & cell types</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=80, r=40, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
# fig.write_html(f"figures/cell_type/{cell_type}_modules.html")
fig.write_image(f"figures/cell_type/{cell_type.split(" ")[0]}_modules.png", width=fig_width, height=800, scale=2)

In [61]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/cell_type/cell_type_{freq_idx}_eigvals.html")
fig.write_image(f"figures/cell_type/{cell_type.split(" ")[0]}_eigvals.png", width=fig_width, height=300, scale=2)

## Alpha

In [62]:
cell_type = "Alpha"

In [63]:
# Decompose weights and extract gene modules
class_idx = type2idx[cell_type]
vals, vecs = decompose_class_weights(model, class_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{cell_type}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Alpha ====================
==================== Module 0 ====================
Positive genes: ['Neurog3' 'Eef1a1' 'Malat1' 'Cd63' 'Mdk' 'Gnas' 'Tmsb4x' 'Cdkn1a'
 'Cyb5r3' 'Krt8']...
Negative genes: ['Sst' 'Hhex' 'Iapp' 'Meis2' 'Arg1' 'Deb1' 'Igfbp5' 'Ccnd1' 'Pcsk2' 'Dlk1']...
==================== Module 1 ====================
Positive genes: ['Nnat' 'Rbp4' 'Iapp' 'Pdx1' 'Mafb' 'Pyy' 'Gng12' '1700086L19Rik' 'Pcsk2'
 'Gadd45a']...
Negative genes: ['Gcg' 'Spp1' 'Krt18' 'Mt1' 'Cldn3' 'Gast' 'Cyr61' 'Tm4sf4' 'Peg3' 'Jun']...
==================== Module 2 ====================
Positive genes: ['Chgb' 'Ttr' 'Spp1' 'Mafb' 'S100a11' 'Tspan7' 'Gcg' 'Pcsk1n' 'Clu'
 'Ubqln2']...
Negative genes: ['Ghrl' 'Cck' 'Sst' 'Arg1' 'Cdkn1a' 'Rbp4' 'Mdk' 'Cd24a' 'Hhex' 'Fxyd3']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1338,Regulation Of Beta-Cell Development R-HSA-186712,PAX4;INSM1;NKX6-1;FOXA3;NEUROG3,Reactome_2022,0.000393
1337,Regulation Of Gene Expression In Endocrine-Com...,PAX4;INSM1;NEUROG3,Reactome_2022,0.000393
1340,Response To Elevated Platelet Cytosolic Ca2+ R...,CD63;SPARC;TTR;TMSB4X;PFN1;CALM1;CLU,Reactome_2022,0.000432
1339,Platelet Degranulation R-HSA-114608,CD63;SPARC;TTR;TMSB4X;PFN1;CALM1;CLU,Reactome_2022,0.000432
1067,Secretory Granule Lumen (GO:0034774),EEF1A1;HSPA8;HSP90AA1;SPARC;CYB5R3;TTR;TMSB4X;...,GO_Cellular_Component_2023,0.000489
1765,Apoptosis,BTG2;CDKN1A;KRT18;GADD45A;CASP6;TXNIP;CLU,MSigDB_Hallmark_2020,0.000674
1766,Pancreas Beta Cells,PAX4;INSM1;NKX6-1;NEUROG3,MSigDB_Hallmark_2020,0.000960
1185,Maturity onset diabetes of the young,PAX4;NKX6-1;FOXA3;NEUROG3,KEGG_2019_Mouse,0.001437
1341,Neutrophil Degranulation R-HSA-6798695,EEF1A1;HSPA8;CD63;HSP90AA1;CYB5R3;TTR;COTL1;CO...,Reactome_2022,0.001907
1068,Ficolin-1-Rich Granule (GO:0101002),EEF1A1;HSPA8;HSP90AA1;COTL1;COMMD3;DYNLL1;CLU,GO_Cellular_Component_2023,0.002286



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1053,Maturity onset diabetes of the young,HHEX;NR5A2;HNF1B;IAPP,KEGG_2019_Mouse,0.001607
1633,Pancreas Beta Cells,PCSK2;SST;IAPP;ISL1,MSigDB_Hallmark_2020,0.001686
1634,Apoptosis,CCND1;ERBB3;GCH1;GPX3;XIAP;MCL1,MSigDB_Hallmark_2020,0.002850
1670,Spinal Cord Injury WP2432,CCND1;ARG1;SOX9;FOS;CD47,WikiPathways_2019_Mouse,0.005085
1669,Amino Acid metabolism WP662,TH;ARG1;SMS;ASNS;HADH,WikiPathways_2019_Mouse,0.005085
1635,Estrogen Response Late,KCNK5;CCND1;TH;SIAH2;PLXNB1;FOS,MSigDB_Hallmark_2020,0.006084
1636,Androgen Response,CCND1;HPGD;ADAMTS1;SMS,MSigDB_Hallmark_2020,0.014382
1637,TNF-alpha Signaling via NF-kB,KLF6;CCND1;GCH1;FOS;MCL1,MSigDB_Hallmark_2020,0.023932
1671,IL-6 signaling Pathway WP387,ERBB3;EIF4EBP1;FOS;SOS1,WikiPathways_2019_Mouse,0.028747
1672,ErbB signaling pathway WP1261,ERBB3;EIF4EBP1;SOS1,WikiPathways_2019_Mouse,0.028747


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
1905,TNF-alpha Signaling via NF-kB,JUN;CDKN1A;TSC22D1;PLK2;SPHK1;ID2;NINJ1;HES1;A...,MSigDB_Hallmark_2020,0.000029
0,Regulation Of Cell Population Proliferation (G...,BEX4;JUN;CDKN1A;TSC22D1;HMGB2;PTN;CLU;FABP3;HH...,GO_Biological_Process_2023,0.000046
1906,Hypoxia,ERRFI1;JUN;LDHA;CDKN1A;CSRP2;ANXA2;CITED2;ENO1,MSigDB_Hallmark_2020,0.000098
1907,Epithelial Mesenchymal Transition,GJA1;JUN;SPARC;ID2;TPM1;SPP1;CAPG;VIM,MSigDB_Hallmark_2020,0.000098
1945,EGFR1 Signaling Pathway WP572,GRB7;ERRFI1;GJA1;JUN;KRT18;KRT8;KRT7;RBBP7,WikiPathways_2019_Mouse,0.000193
1908,Cholesterol Homeostasis,ERRFI1;FABP5;ANXA5;CD9;CLU,MSigDB_Hallmark_2020,0.000346
1909,p53 Pathway,JUN;CDKN1A;TSC22D1;PLK2;SPHK1;NINJ1;S100A10,MSigDB_Hallmark_2020,0.000526
1910,Apoptosis,JUN;CDKN1A;KRT18;GPX3;HMGB2;CLU,MSigDB_Hallmark_2020,0.001056
1,Negative Regulation Of Nucleic Acid-Templated ...,JUN;CENPF;HHEX;CITED2;ID2;HMGB2;BIRC5;HES1;RBB...,GO_Biological_Process_2023,0.001181
2,Negative Regulation Of Programmed Cell Death (...,KRT18;TSC22D1;CITED2;PLK2;SPHK1;ANXA5;CDK1;BIR...,GO_Biological_Process_2023,0.001181


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1574,Pancreas Beta Cells,PCSK2;CHGA;MAFB;PDX1;PAX6;GCG;IAPP;SYT13;INSM1...,MSigDB_Hallmark_2020,1.704110e-13
1039,Maturity onset diabetes of the young,INS1;INS2;PDX1;PAX6;IAPP;NKX6-1;MNX1,KEGG_2019_Mouse,8.518797e-09
1205,Response To Elevated Platelet Cytosolic Ca2+ R...,CAP1;SPARC;TTR;ACTN1;ANXA5;SCG3;TAGLN2;CLU,Reactome_2022,5.349255e-05
1204,Platelet Degranulation R-HSA-114608,CAP1;SPARC;TTR;ACTN1;ANXA5;SCG3;TAGLN2;CLU,Reactome_2022,5.349255e-05
1206,"Platelet Activation, Signaling And Aggregation...",RAP1B;CAP1;SPARC;TTR;ACTN1;ANXA5;SCG3;TAGLN2;C...,Reactome_2022,7.166697e-05
1207,Peptide Hormone Metabolism R-HSA-2980736,PCSK2;CHGA;SLC30A8;CPE;PAX6;GCG,Reactome_2022,5.162294e-04
1575,KRAS Signaling Up,PCSK1N;MAFB;TSPAN7;SPP1;CPE;SCG3;GADD45G,MSigDB_Hallmark_2020,1.215549e-03
1208,Hemostasis R-HSA-109582,RAP1B;CAP1;SPARC;TTR;ACTN1;TSPAN7;ANXA5;SCG3;T...,Reactome_2022,2.247907e-03
1209,Regulation Of Beta-Cell Development R-HSA-186712,PAX6;IAPP;INSM1;NKX6-1,Reactome_2022,3.187519e-03
1040,Type II diabetes mellitus,INS1;INS2;PDX1;MAPK3,KEGG_2019_Mouse,6.829789e-03



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1596,Pancreas Beta Cells,SST;STXBP1;ISL1;NEUROG3,MSigDB_Hallmark_2020,0.001686
1035,Lysosome,GNPTG;CD63;ATP6V0B;NPC2;IDS;CTSF,KEGG_2019_Mouse,0.005477
1598,Epithelial Mesenchymal Transition,JUN;ANPEP;APLP1;PPIB;SCG2;MEST,MSigDB_Hallmark_2020,0.006084
1597,mTORC1 Signaling,BTG2;CDKN1A;HSPA5;PSMC2;RPN1;HSP90B1,MSigDB_Hallmark_2020,0.006084
937,Axon (GO:0030424),OLFM1;TH;TUBB3;MAP1B;CCK;GHRL;CD200,GO_Cellular_Component_2023,0.007520
812,Calcium-Dependent Phospholipase A2 Activity (G...,PLA2G2F;PLA2G12A;PLA2G2D,GO_Molecular_Function_2023,0.008098
0,Axon Development (GO:0061564),NEUROD2;MAP1B;APLP1;CCK;ISL1;NEUROG3,GO_Biological_Process_2023,0.008394
1184,Acyl Chain Remodelling Of PG R-HSA-1482925,PLA2G2F;PLA2G12A;PLA2G2D,Reactome_2022,0.012932
1183,Peptide Hormone Metabolism R-HSA-2980736,ANPEP;FFAR4;GHRL;MBOAT4;ISL1,Reactome_2022,0.012932
1182,Acyl Chain Remodelling Of PI R-HSA-1482922,PLA2G2F;PLA2G12A;PLA2G2D,Reactome_2022,0.012932


In [64]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

class_idx = type2idx[cell_type]

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
cell_type_labels = [class_names[i] for i in labels_np]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], cell_type_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Cell Type': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_cell_types = sorted(plot_df['Cell Type'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_cell_types):
        ctype_scores = module_df[module_df['Cell Type'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Cell Type: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Cell Type Distributions Along Top 3 Modules for {cell_type} Cells"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & cell types</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=80, r=40, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
# fig.write_html(f"figures/cell_type/{cell_type}_modules.html")
fig.write_image(f"figures/cell_type/{cell_type}_modules.png", width=fig_width, height=800, scale=2)

In [65]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/cell_type/cell_type_{freq_idx}_eigvals.html")
fig.write_image(f"figures/cell_type/{cell_type}_eigvals.png", width=fig_width, height=300, scale=2)

## Beta

In [66]:
cell_type = "Beta"

In [67]:
# Decompose weights and extract gene modules
class_idx = type2idx[cell_type]
vals, vecs = decompose_class_weights(model, class_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"{cell_type}", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Beta ====================
==================== Module 0 ====================
Positive genes: ['Ghrl' 'Cdkn1a' 'Rbp4' 'Cck' 'Isl1' 'Mdk' 'Neurog3' 'Malat1' 'Maged2'
 'Cd63']...
Negative genes: ['Chgb' 'Nnat' 'Pdx1' 'Ociad2' 'Mafb' 'Ins2' 'Gng12' 'Nkx6-1' 'Dlk1'
 'Pcsk2']...
==================== Module 1 ====================
Positive genes: ['Rbp4' 'Pyy' 'Isl1' 'Chgb' 'Chga' 'Pcsk1n' 'Iapp' 'Fam183b' 'Ghrl' 'Gch1']...
Negative genes: ['Neurog3' 'Spp1' 'Mdk' 'Btbd17' 'Gadd45a' 'Sox4' 'Serpinh1' 'Tmsb4x'
 'Tead2' 'Cdk4']...
==================== Module 2 ====================
Positive genes: ['Iapp' 'Pyy' 'Gcg' 'Ghrl' 'Tmem27' 'Scgn' 'Nnat' 'Slc38a5' 'Tmsb4x'
 'Selm']...
Negative genes: ['Prrg2' 'Cd24a' 'Hhex' 'Akr1c19' 'Ier2' 'Emb' 'Npepl1' 'Cryba2' 'Sst'
 'Jun']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1233,Protein processing in endoplasmic reticulum,PDIA3;HSPA8;HSP90AA1;HSPA5;SSR4;RPN2;SSR2;RPN1...,KEGG_2019_Mouse,0.000001
1098,Secretory Granule Lumen (GO:0034774),EEF1A1;HSPA8;HSP90AA1;CYB5R3;NPC2;ARG1;TMSB4X;...,GO_Cellular_Component_2023,0.000071
1099,Endocytic Vesicle Lumen (GO:0071682),HSP90AA1;CTSL;CALR;HSP90B1,GO_Cellular_Component_2023,0.000223
1387,Neutrophil Degranulation R-HSA-6798695,HSPA8;CD63;HSP90AA1;ATP6AP2;CTSZ;DYNLL1;EEF1A1...,Reactome_2022,0.000247
1234,Antigen processing and presentation,PDIA3;HSPA8;HSP90AA1;HSPA5;CTSL;CALR,KEGG_2019_Mouse,0.000459
1100,Intracellular Organelle Lumen (GO:0070013),PDIA3;HSPA8;CD63;HSP90AA1;HSPA5;CTSZ;TXNDC12;H...,GO_Cellular_Component_2023,0.001000
1388,Peptide Hormone Metabolism R-HSA-2980736,ANPEP;CTSZ;ATP6AP2;GHRL;MBOAT4;ISL1,Reactome_2022,0.001155
1101,Ficolin-1-Rich Granule (GO:0101002),EEF1A1;HSPA8;HSP90AA1;ATP6AP2;CTSZ;COTL1;DYNLL1,GO_Cellular_Component_2023,0.001161
1102,Lysosome (GO:0005764),HSPA8;GNPTG;CD63;ATP6V0B;HSP90AA1;IFITM2;NPC2;...,GO_Cellular_Component_2023,0.001161
1389,ATF6 (ATF6-alpha) Activates Chaperone Genes R-...,HSPA5;CALR;HSP90B1,Reactome_2022,0.001175



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
0,Regulation Of Insulin Secretion (GO:0050796),G6PC2;SLC30A8;NNAT;PDX1;FFAR1;CLOCK;ADRA2A;GIP,GO_Biological_Process_2023,0.000010
1051,Maturity onset diabetes of the young,INS1;INS2;PDX1;NKX6-1;MNX1,KEGG_2019_Mouse,0.000029
1673,Pancreas Beta Cells,PCSK2;G6PC2;MAFB;PDX1;NKX6-1,MSigDB_Hallmark_2020,0.000065
1,Positive Regulation Of Protein Secretion (GO:0...,SLC30A8;NNAT;PDX1;FFAR1;EZR;SYTL4,GO_Biological_Process_2023,0.000886
1052,Insulin secretion,INS1;INS2;PDX1;FFAR1;GIP,KEGG_2019_Mouse,0.005034
2,Positive Regulation Of Cytoplasmic Translation...,SYNCRIP;IGF2BP1;YBX3,GO_Biological_Process_2023,0.009377
3,Positive Regulation Of Insulin Secretion (GO:0...,SLC30A8;NNAT;PDX1;FFAR1,GO_Biological_Process_2023,0.009377
1713,Fatty Acid Biosynthesis WP336,ACAA2;ECH1;HADH,WikiPathways_2019_Mouse,0.010807
4,Glucose Homeostasis (GO:0042593),MLXIPL;G6PC2;SLC30A8;CARTPT;ADRA2A,GO_Biological_Process_2023,0.010909
5,Positive Regulation Of Peptide Hormone Secreti...,SLC30A8;NNAT;PDX1;FFAR1,GO_Biological_Process_2023,0.010909


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
1503,Epithelial Mesenchymal Transition,SPARC;GADD45A;TPM1;SPP1;SERPINH1;CAPG;VIM;PPIB,MSigDB_Hallmark_2020,0.000317
1157,Maturity onset diabetes of the young,PAX4;NKX6-1;FOXA3;NEUROG3,KEGG_2019_Mouse,0.000907
1504,E2F Targets,RANBP1;CDK4;HMGB2;HMGB3;PAICS;RAN;GSPT1,MSigDB_Hallmark_2020,0.001413
1505,Apoptosis,BTG2;GADD45A;CASP6;HMGB2;TXNIP;CLU,MSigDB_Hallmark_2020,0.002269
1506,p53 Pathway,GPX2;BTG2;TRAF4;GADD45A;TXNIP;S100A10,MSigDB_Hallmark_2020,0.005451
1507,Pancreas Beta Cells,PAX4;NKX6-1;NEUROG3,MSigDB_Hallmark_2020,0.009010
1057,Collagen-Containing Extracellular Matrix (GO:0...,SPARC;IGFBPL1;ANXA2;AMBP;MDK;SERPINH1;COL9A3;C...,GO_Cellular_Component_2023,0.010591
1253,Regulation Of Beta-Cell Development R-HSA-186712,PAX4;NKX6-1;FOXA3;NEUROG3,Reactome_2022,0.012922
920,Glutathione Transferase Activity (GO:0004364),GSTA3;MGST1;SH3BGRL3,GO_Molecular_Function_2023,0.016811
919,Protein Phosphatase Inhibitor Activity (GO:000...,PPP1R14A;PPP1R1B;PTN,GO_Molecular_Function_2023,0.016811


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1197,Pancreas Beta Cells,PCSK2;SCGN;GCG;IAPP;NEUROG3,MSigDB_Hallmark_2020,0.000052
814,Protein export,IMMP1L;HSPA5;SEC61B;SEC11C,KEGG_2019_Mouse,0.001068
815,Protein processing in endoplasmic reticulum,HSPA5;SSR4;CALR;SEC61B;PDIA6;HSP90B1;PDIA4,KEGG_2019_Mouse,0.001068
935,ATF6 (ATF6-alpha) Activates Chaperones R-HSA-3...,HSPA5;CALR;HSP90B1,Reactome_2022,0.002556
934,ATF6 (ATF6-alpha) Activates Chaperone Genes R-...,HSPA5;CALR;HSP90B1,Reactome_2022,0.002556
937,Response To Elevated Platelet Cytosolic Ca2+ R...,TTR;HSPA5;TMSB4X;SCG3;MAGED2;TUBA4A,Reactome_2022,0.003194
936,Platelet Degranulation R-HSA-114608,TTR;HSPA5;TMSB4X;SCG3;MAGED2;TUBA4A,Reactome_2022,0.003194
938,Peptide Hormone Metabolism R-HSA-2980736,PCSK2;CPE;GCG;GHRL;SEC11C,Reactome_2022,0.004424
939,"Synthesis, Secretion, And Deacylation Of Ghrel...",GCG;GHRL;SEC11C,Reactome_2022,0.004860
1198,mTORC1 Signaling,SDF2L1;HSPA5;CALR;GAPDH;TUBA4A;HSP90B1,MSigDB_Hallmark_2020,0.008112



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
889,Transcription Regulatory Region Nucleic Acid B...,EGR1;JUN;SFPQ;HHEX;HMGB2;FOS;SOX4;FOXA2,GO_Molecular_Function_2023,0.002418
1508,TNF-alpha Signaling via NF-kB,EGR1;MARCKS;JUN;BTG2;FOS;JUNB;IER2,MSigDB_Hallmark_2020,0.002431
1547,Oxidative Stress WP412,MGST1;FOS;JUNB,WikiPathways_2019_Mouse,0.007256
1546,Novel Jun-Dmp1 Pathway WP3654,JUN;FOS;JUNB,WikiPathways_2019_Mouse,0.007256
1545,Spinal Cord Injury WP2432,EGR1;BTG2;SOX9;FOS;VIM,WikiPathways_2019_Mouse,0.007256
890,Transcription Cis-Regulatory Region Binding (G...,EGR1;JUN;SFPQ;HHEX;HMGB2;SOX9;FOS;ISL1;SOX4;FOXA2,GO_Molecular_Function_2023,0.009433
1548,Signaling of Hepatocyte Growth Factor Receptor...,RAP1B;JUN;FOS,WikiPathways_2019_Mouse,0.009727
891,Sequence-Specific Double-Stranded DNA Binding ...,EGR1;JUN;SFPQ;HHEX;FEV;HMGB2;SOX9;FOS;ISL1;JUN...,GO_Molecular_Function_2023,0.011266
1509,Pancreas Beta Cells,SST;ISL1;FOXA2,MSigDB_Hallmark_2020,0.015944
1510,Apoptosis,APP;JUN;BTG2;KRT18;HMGB2,MSigDB_Hallmark_2020,0.015944


In [68]:
# Plot histograms of cell type distributions along top 3 eigencomponents (modules 1–3)

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

class_idx = type2idx[cell_type]

# Prepare full dataset (train + val) if not already loaded
if 'full_data' not in globals() or 'full_labels' not in globals():
    full_loader = torch.utils.data.DataLoader(
        torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
        batch_size=len(train_dataset) + len(val_dataset),
        shuffle=False,
    )
    with torch.no_grad():
        full_batch = next(iter(full_loader))
        full_data, full_labels = full_batch

# Project all cells onto the first 3 eigencomponents (modules)
with torch.no_grad():
    top_k = 3
    component_scores = torch.matmul(full_data, vecs[:, :top_k])  # (n_cells, 3)

# Convert to numpy / pandas for plotting
scores_np = component_scores.numpy()
labels_np = full_labels.numpy()
class_names = adata.obs[class_key].cat.categories
cell_type_labels = [class_names[i] for i in labels_np]

# Build a long-form DataFrame for Plotly
score_rows = []
for m in range(top_k):
    for score, ctype in zip(scores_np[:, m], cell_type_labels):
        score_rows.append({
            'Module': f'Module {m+1}',
            'Score': score,
            'Cell Type': ctype,
        })
plot_df = pd.DataFrame(score_rows)

# Establish consistent (shared) bin edges across ALL histograms (modules & cell types)
# Use symmetric range around zero for fair comparison of signed projections.
extent = max(abs(scores_np.min()), abs(scores_np.max()))
# Add a small epsilon to avoid zero-width issues when extremely narrow
epsilon = 1e-9
global_min, global_max = -extent - epsilon, extent + epsilon
n_bins = 100  # adjust if you want coarser/finer resolution
bin_size = (global_max - global_min) / n_bins

# Create stacked subplots (one histogram per module)
fig = make_subplots(rows=3, cols=1, shared_xaxes=False, shared_yaxes=False,
                    vertical_spacing=0.07, subplot_titles=[f'Module {i+1}' for i in range(top_k)])

unique_cell_types = sorted(plot_df['Cell Type'].unique())

opacity = 0.75
for m in range(top_k):
    module_name = f'Module {m+1}'
    module_df = plot_df[plot_df['Module'] == module_name]
    row = m + 1
    for idx, ctype in enumerate(unique_cell_types):
        ctype_scores = module_df[module_df['Cell Type'] == ctype]['Score']
        # Use predefined cluster color; fall back to palette if missing
        marker_color = cluster_colors.get(ctype, px.colors.qualitative.Plotly[idx % len(px.colors.qualitative.Plotly)])
        fig.add_trace(
            go.Histogram(
                x=ctype_scores,
                name=ctype if m == 0 else ctype,
                legendgroup=ctype,
                opacity=opacity,
                # consistent bin edges for ALL traces
                xbins=dict(start=global_min, end=global_max, size=bin_size),
                marker_line=dict(width=0),
                marker_color=marker_color,
                showlegend=(m == 0),
                histnorm='probability',  # per-cell-type normalized
                hovertemplate=(
                    f"{module_name}<br>Cell Type: {ctype}<br>Bin: %{{x:.3f}}<br>Normalized Count: %{{y:.3f}}<extra></extra>"
                ),
            ),
            row=row, col=1
        )

# Layout adjustments
fig.update_layout(
    barmode='overlay',
    title=(
        f"Cell Type Distributions Along Top 3 Modules for {cell_type} Cells"\
        + "<br><span style='font-size:0.85em;'>Shared symmetric bin edges across all modules & cell types</span>"
    ),
    height=250 * top_k + 170,
    width=fig_width,
    margin=dict(l=80, r=40, t=110, b=60),
    legend=dict(title=None, orientation='v', yanchor='top', y=0.98, xanchor='left', x=1.02),
)

# Axis titles
for m in range(top_k):
    row = m + 1
    fig.update_yaxes(title_text='Count (normalized)', row=row, col=1)
    if row == top_k:
        fig.update_xaxes(title_text='Module Value', row=row, col=1)

# Apply font size customization
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize), title=None),
)

# Tune axis title & tick font
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save outputs
# fig.write_html(f"figures/cell_type/{cell_type}_modules.html")
fig.write_image(f"figures/cell_type/{cell_type}_modules.png", width=fig_width, height=800, scale=2)

In [69]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/cell_type/cell_type_{freq_idx}_eigvals.html")
fig.write_image(f"figures/cell_type/{cell_type}_eigvals.png", width=fig_width, height=300, scale=2)